<a href="https://colab.research.google.com/github/SithmiDesilva/Hallucination-Aware-AI-Framework-for-Trustworthy-Supply-Chain-Decision-Intelligence.-/blob/News_data_scraping/News_Data_Extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Searching for Supply Chain News using Exa API

First, we need to install the official `exa_py` SDK.

In [1]:
!pip install exa_py

In [2]:
import os, json, time, re, datetime as dt
from urllib.parse import urlparse
import pandas as pd
from exa_py import Exa

# --- API key: Colab Secrets first, then environment variable ---
try:
    from google.colab import userdata
    exa_api_key = userdata.get('EXA_API_KEY')
except Exception:
    exa_api_key = os.environ.get('EXA_API_KEY', 'YOUR_EXA_API_KEY')

exa = Exa(api_key=exa_api_key)

# --- Save to Google Drive so files survive the Colab session ---
USE_DRIVE = True
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        OUT_DIR = '/content/drive/MyDrive/IRP/news_data'
    except Exception:
        OUT_DIR = 'news_data'
else:
    OUT_DIR = 'news_data'
os.makedirs(OUT_DIR, exist_ok=True)

# Every run gets an ID so results are reproducible / comparable later
RUN_ID = dt.datetime.now(dt.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
RAW_LOG = os.path.join(OUT_DIR, f'raw_exa_log_{RUN_ID}.jsonl')
SEARCH_TYPE = 'neural'   # if Exa rejects this, use 'auto'
print('Run ID:', RUN_ID, '| output folder:', OUT_DIR)

Mounted at /content/drive
Run ID: 20261125T132210Z | output folder: /content/drive/MyDrive/IRP/news_data


In [3]:
# M5: 2011-01-29 to 2016-06-19 (CA, TX, WI)
# DataCo: orders ~Jan 2015 to Jan 2018 (global markets) - check against your copy:
#   pd.to_datetime(dataco['order date (DateOrders)']).agg(['min', 'max'])
DATASET_WINDOWS = {
    'M5':     ('2011-01-29', '2016-06-19'),
    'DataCo': ('2015-01-01', '2018-01-31'),
}
PUBLISH_BUFFER_DAYS = 30   # catch reporting published shortly after the window ends

def iso(d, end=False):
    return f'{d}T23:59:59.000Z' if end else f'{d}T00:00:00.000Z'

def yearly_windows(start, end):
    """Split a date range into calendar-year windows (one Exa search per year)."""
    s, e = pd.Timestamp(start), pd.Timestamp(end)
    out = []
    for y in range(s.year, e.year + 1):
        ws = max(s, pd.Timestamp(f'{y}-01-01'))
        we = min(e, pd.Timestamp(f'{y}-12-31'))
        out.append((ws.strftime('%Y-%m-%d'), we.strftime('%Y-%m-%d')))
    return out

def norm_url(u):
    """Normalise a URL so search hits and fetched contents match (http/https, www, trailing slash)."""
    if not isinstance(u, str):
        return None
    p = urlparse(u.strip())
    host = p.netloc.lower().removeprefix('www.')
    path = p.path.rstrip('/')
    return f'{host}{path}' + (f'?{p.query}' if p.query else '')

print(yearly_windows(*DATASET_WINDOWS['M5']))
print(yearly_windows(*DATASET_WINDOWS['DataCo']))

[('2011-01-29', '2011-12-31'), ('2012-01-01', '2012-12-31'), ('2013-01-01', '2013-12-31'), ('2014-01-01', '2014-12-31'), ('2015-01-01', '2015-12-31'), ('2016-01-01', '2016-06-19')]
[('2015-01-01', '2015-12-31'), ('2016-01-01', '2016-12-31'), ('2017-01-01', '2017-12-31'), ('2018-01-01', '2018-01-31')]


In [4]:
def result_to_dict(r):
    return {
        'id': getattr(r, 'id', None),
        'url': getattr(r, 'url', None),
        'title': getattr(r, 'title', None),
        'published_date': getattr(r, 'published_date', None),
        'author': getattr(r, 'author', None),
        'score': getattr(r, 'score', None),
        'text': getattr(r, 'text', None),
    }

def log_raw(kind, params, results):
    """Append the raw Exa response to a JSONL log (needed for reproducibility in the thesis)."""
    with open(RAW_LOG, 'a', encoding='utf-8') as f:
        f.write(json.dumps({
            'kind': kind,
            'run_id': RUN_ID,
            'logged_at': dt.datetime.now(dt.timezone.utc).isoformat(),
            'params': params,
            'results': [result_to_dict(r) for r in results],
        }, ensure_ascii=False, default=str) + '\n')

def with_retry(fn, *args, retries=3, wait=2, **kwargs):
    for attempt in range(1, retries + 1):
        try:
            return fn(*args, **kwargs)
        except Exception as e:
            if attempt == retries:
                raise
            print(f'   retry {attempt}/{retries - 1} after error: {e}')
            time.sleep(wait * attempt)

def run_search(dataset, category, query, start, end, num_results=10):
    params = dict(
        query=query,
        num_results=num_results,
        type=SEARCH_TYPE,
        start_published_date=iso(start),
        end_published_date=iso(end, end=True),
    )
    resp = with_retry(exa.search, **params)
    log_raw('search', {**params, 'dataset': dataset, 'category': category}, resp.results)
    rows = []
    for rank, r in enumerate(resp.results, 1):
        rows.append({
            'dataset': dataset,
            'category': category,
            'query': query,
            'window_start': start,
            'window_end': end,
            'rank': rank,
            'url': r.url,
            'title': r.title,
            'published_date': getattr(r, 'published_date', None),
            'score': getattr(r, 'score', None),
        })
    return rows

In [5]:
# No year lists inside the query text - the date filter does that job.

# --- Topic queries: run once per calendar year inside the dataset window ---
m5_topic_queries = {
    'M5_CA_Weather': 'severe storms, floods, drought or heatwave in California affecting stores, shoppers or supplies',
    'M5_TX_Weather': 'severe weather in Texas such as floods, winter storms, heatwaves or hurricanes affecting retail and shoppers',
    'M5_WI_Weather': 'Wisconsin blizzard, snowstorm or flooding disrupting shopping and stores',
    'M5_SNAP': 'changes to SNAP food stamp benefits or payment schedule affecting grocery spending',
    'M5_Walmart_Stores': 'Walmart store openings, closures or remodels in California, Texas or Wisconsin',
    'M5_Walmart_Pricing': 'Walmart price rollbacks, price cuts, promotions and price matching',
    'M5_Food_Recalls': 'US food product recall due to contamination announced by FDA or USDA',
    'M5_Household_Hobby_Recalls': 'CPSC recall of household products, toys or hobby products sold at major US retailers',
    'M5_Local_Economy': 'plant closure or mass layoffs in California, Texas or Wisconsin hitting the local economy',
    'M5_Fuel_Prices': 'US gasoline price spike or drop affecting consumer spending',
}

# DataCo is GLOBAL (USCA, Europe, LATAM, Pacific Asia, Africa) - not only US queries
dataco_topic_queries = {
    'DataCo_Port_Congestion': 'port congestion and vessel delays at major container ports',
    'DataCo_Port_Labour': 'dockworker strike or port labour dispute halting cargo',
    'DataCo_Carrier_Disruption': 'container shipping line disruption, bankruptcy or poor schedule reliability',
    'DataCo_Trucking_Capacity': 'trucking capacity crunch, driver shortage and rising freight rates',
    'DataCo_Weather_Logistics': 'hurricane, typhoon or flooding disrupting ports, freight and deliveries',
    'DataCo_Trade_Customs': 'customs delays, new tariffs or trade policy changes disrupting imports and exports',
    'DataCo_Supplier_Shortage': 'factory shutdown or component shortage delaying manufacturer shipments',
    'DataCo_Europe': 'freight and logistics disruption in Europe such as strikes, border delays or port closures',
    'DataCo_LATAM': 'logistics disruption in Latin America such as trucking strikes, port problems or customs delays in Brazil or Mexico',
    'DataCo_Asia_Pacific': 'supply chain disruption in Asia Pacific such as China factory closures, port shutdowns or typhoons',
    'DataCo_Africa': 'port congestion and logistics delays at African ports and trade corridors',
    'DataCo_Ecommerce_Delivery': 'online retail delivery delays and late shipments during peak season',
}

# --- Known events with tight windows: (dataset, category, query, start, end) ---
event_queries = [
    ('M5', 'EVT_TX_Drought_Heat_2011', 'Texas 2011 record drought and heatwave', '2011-05-01', '2011-12-31'),
    ('M5', 'EVT_SNAP_Cut_2013', 'SNAP benefit cut November 2013 as stimulus boost expires', '2013-09-01', '2014-01-31'),
    ('M5', 'EVT_Polar_Vortex_2014', 'polar vortex extreme cold in the Midwest and Wisconsin, January 2014', '2014-01-01', '2014-03-31'),
    ('M5', 'EVT_CA_Drought_2014_2015', 'California drought emergency and water restrictions, impact on food prices', '2014-01-01', '2015-12-31'),
    ('M5', 'EVT_Blue_Bell_Recall_2015', 'Blue Bell ice cream listeria recall', '2015-03-01', '2015-08-31'),
    ('M5', 'EVT_TX_Memorial_Day_Floods_2015', 'Texas Memorial Day floods in May 2015 in Houston and Austin', '2015-05-20', '2015-07-15'),
    ('M5', 'EVT_Gas_Price_Collapse_2015', 'gas prices fall below $2 as oil prices collapse, effect on consumers', '2014-11-01', '2016-03-31'),
    ('M5', 'EVT_Walmart_Closures_2016', 'Walmart announces closure of 269 stores in January 2016', '2016-01-01', '2016-03-31'),
    ('DataCo', 'EVT_West_Coast_Port_Slowdown', 'US West Coast ports slowdown, ILWU contract dispute and cargo backlog', '2014-11-01', '2015-04-30'),
    ('DataCo', 'EVT_Tianjin_Explosion_2015', 'Tianjin port explosion impact on shipping and supply chains', '2015-08-12', '2015-11-30'),
    ('DataCo', 'EVT_France_Strikes_2016', 'France strikes and blockades at ports and fuel depots over labour law', '2016-05-01', '2016-07-15'),
    ('DataCo', 'EVT_Panama_Canal_Expansion_2016', 'expanded Panama Canal opens to neopanamax ships', '2016-06-01', '2016-09-30'),
    ('DataCo', 'EVT_Brexit_Vote_2016', 'Brexit vote impact on supply chains and logistics', '2016-06-23', '2016-12-31'),
    ('DataCo', 'EVT_Hanjin_Bankruptcy_2016', 'Hanjin Shipping bankruptcy strands cargo ships', '2016-08-31', '2016-12-31'),
    ('DataCo', 'EVT_India_Demonetisation_2016', 'India demonetisation impact on trucking and logistics', '2016-11-08', '2017-02-28'),
    ('DataCo', 'EVT_NotPetya_Maersk_2017', 'NotPetya cyberattack shuts down Maersk port terminals', '2017-06-27', '2017-09-30'),
    ('DataCo', 'EVT_Harvey_Irma_Maria_2017', 'Hurricanes Harvey, Irma and Maria disrupt freight and ports', '2017-08-25', '2017-11-30'),
    ('DataCo', 'EVT_Mexico_Earthquake_2017', 'September 2017 Mexico earthquake disrupts factories and logistics', '2017-09-07', '2017-11-30'),
    ('DataCo', 'EVT_ELD_Mandate_2017', 'ELD mandate trucking capacity crunch and spot rate surge, December 2017', '2017-10-01', '2018-02-28'),
]

In [6]:
NUM_RESULTS = 10   # per search; raise to 20-25 if you want a bigger corpus

jobs = []
for dataset, topics in [('M5', m5_topic_queries), ('DataCo', dataco_topic_queries)]:
    start, end = DATASET_WINDOWS[dataset]
    end = (pd.Timestamp(end) + pd.Timedelta(days=PUBLISH_BUFFER_DAYS)).strftime('%Y-%m-%d')
    for category, q in topics.items():
        for ws, we in yearly_windows(start, end):
            jobs.append((dataset, category, q, ws, we))
jobs += event_queries
print(f'{len(jobs)} searches to run\n')

search_rows, failed_searches = [], []
for i, (dataset, category, q, ws, we) in enumerate(jobs, 1):
    try:
        rows = run_search(dataset, category, q, ws, we, num_results=NUM_RESULTS)
        search_rows.extend(rows)
        print(f'[{i}/{len(jobs)}] {category} {ws} -> {we}: {len(rows)} hits')
    except Exception as e:
        failed_searches.append((dataset, category, q, ws, we, str(e)))
        print(f'[{i}/{len(jobs)}] FAILED {category} {ws} -> {we}: {e}')

search_df = pd.DataFrame(search_rows)
search_df.to_csv(os.path.join(OUT_DIR, f'search_hits_{RUN_ID}.csv'), index=False)
print(f'\nSaved {len(search_df)} hits | failed searches: {len(failed_searches)}')

127 searches to run

[1/127] M5_CA_Weather 2011-01-29 -> 2011-12-31: 10 hits
[2/127] M5_CA_Weather 2012-01-01 -> 2012-12-31: 10 hits
[3/127] M5_CA_Weather 2013-01-01 -> 2013-12-31: 10 hits
[4/127] M5_CA_Weather 2014-01-01 -> 2014-12-31: 10 hits
[5/127] M5_CA_Weather 2015-01-01 -> 2015-12-31: 10 hits
[6/127] M5_CA_Weather 2016-01-01 -> 2016-07-19: 10 hits
[7/127] M5_TX_Weather 2011-01-29 -> 2011-12-31: 10 hits
[8/127] M5_TX_Weather 2012-01-01 -> 2012-12-31: 10 hits
[9/127] M5_TX_Weather 2013-01-01 -> 2013-12-31: 10 hits
[10/127] M5_TX_Weather 2014-01-01 -> 2014-12-31: 10 hits
[11/127] M5_TX_Weather 2015-01-01 -> 2015-12-31: 10 hits
[12/127] M5_TX_Weather 2016-01-01 -> 2016-07-19: 10 hits
[13/127] M5_WI_Weather 2011-01-29 -> 2011-12-31: 10 hits
[14/127] M5_WI_Weather 2012-01-01 -> 2012-12-31: 10 hits
[15/127] M5_WI_Weather 2013-01-01 -> 2013-12-31: 10 hits
[16/127] M5_WI_Weather 2014-01-01 -> 2014-12-31: 10 hits
[17/127] M5_WI_Weather 2015-01-01 -> 2015-12-31: 10 hits
[18/127] M5_WI_Weat

In [7]:
print('Total hits:', len(search_df), '| unique URLs:', search_df['url'].nunique())

display(search_df.groupby(['dataset', 'category'])
        .agg(hits=('url', 'size'), unique_urls=('url', 'nunique')))

# Coverage per publication year - look for empty years
pub_year = pd.to_datetime(search_df['published_date'], errors='coerce', utc=True).dt.year
display(search_df.assign(pub_year=pub_year)
        .groupby(['pub_year', 'dataset']).size().unstack(fill_value=0))

Total hits: 1270 | unique URLs: 1228


hits  unique_urls
dataset category                                          
DataCo  DataCo_Africa                      40           40
        DataCo_Asia_Pacific                40           40
        DataCo_Carrier_Disruption          40           40
        DataCo_Ecommerce_Delivery          40           40
        DataCo_Europe                      40           40
        DataCo_LATAM                       40           40
        DataCo_Port_Congestion             40           40
        DataCo_Port_Labour                 40           40
        DataCo_Supplier_Shortage           40           40
        DataCo_Trade_Customs               40           40
        DataCo_Trucking_Capacity           40           40
        DataCo_Weather_Logistics           40           40
        EVT_Brexit_Vote_2016               10           10
        EVT_ELD_Mandate_2017               10           10
        EVT_France_Strikes_2016            10           10
        EVT_Hanjin_Bankruptcy_2016         10           10
        EVT_Harvey_Irma_Maria_2017         10           10
        EVT_India_Demonetisation_2016      10           10
        EVT_Mexico_Earthquake_2017         10           10
        EVT_NotPetya_Maersk_2017           10           10
        EVT_Panama_Canal_Expansion_2016    10           10
        EVT_Tianjin_Explosion_2015         10           10
        EVT_West_Coast_Port_Slowdown       10           10
M5      EVT_Blue_Bell_Recall_2015          10           10
        EVT_CA_Drought_2014_2015           10           10
        EVT_Gas_Price_Collapse_2015        10           10
        EVT_Polar_Vortex_2014              10           10
        EVT_SNAP_Cut_2013                  10           10
        EVT_TX_Drought_Heat_2011           10           10
        EVT_TX_Memorial_Day_Floods_2015    10           10
        EVT_Walmart_Closures_2016          10           10
        M5_CA_Weather                      60           60
        M5_Food_Recalls                    60           60
        M5_Fuel_Prices                     60           60
        M5_Household_Hobby_Recalls         60           60
        M5_Local_Economy                   60           60
        M5_SNAP                            60           60
        M5_TX_Weather                      60           60
        M5_WI_Weather                      60           60
        M5_Walmart_Pricing                 60           60
        M5_Walmart_Stores                  60           60

dataset,DataCo,M5
pub_year,,
2011,0,110
2012,0,100
2013,0,110
2014,0,121
2015,140,129
2016,169,110
2017,159,0
2018,122,0


In [8]:
# Fixes: no 2000-character cut, keeps published_date and author, tracks failed URLs.
unique_urls = search_df['url'].dropna().drop_duplicates().tolist()
CHUNK = 10

def fetch_contents(urls, **extra):
    rows, got = [], set()
    for i in range(0, len(urls), CHUNK):
        chunk = urls[i:i + CHUNK]
        try:
            resp = with_retry(exa.get_contents, chunk, text=True, **extra)
            log_raw('contents', {'urls': chunk, **extra}, resp.results)
            for d in resp.results:
                text = getattr(d, 'text', None) or ''
                rows.append({
                    'url': d.url,
                    'content_title': getattr(d, 'title', None),
                    'content_published_date': getattr(d, 'published_date', None),
                    'author': getattr(d, 'author', None),
                    'text': text,
                })
                if text.strip():
                    got.add(norm_url(d.url))
        except Exception as e:
            print(f'   chunk {i // CHUNK + 1} failed: {e}')
        print(f'Chunk {i // CHUNK + 1}/{-(-len(urls) // CHUNK)} done')
    missing = [u for u in urls if norm_url(u) not in got]
    return rows, missing

content_rows, missing = fetch_contents(unique_urls)
print(f'\nFirst pass: {len(content_rows)} docs, {len(missing)} with no text')

# Second pass: ask Exa to live-crawl the ones that came back empty
if missing:
    try:
        retry_rows, missing = fetch_contents(missing, livecrawl='fallback')
        content_rows += retry_rows
    except TypeError:
        print('This exa_py version has no livecrawl option - skipping second pass')
    print(f'After retry: {len(missing)} URLs still have no text')

contents_df = pd.DataFrame(content_rows)
contents_df['norm'] = contents_df['url'].map(norm_url)
# keep the longest text if a URL was fetched twice
contents_df = (contents_df.assign(_len=contents_df['text'].str.len())
               .sort_values('_len', ascending=False)
               .drop_duplicates('norm').drop(columns=['_len', 'url']))

Chunk 1/123 done
Chunk 2/123 done
Chunk 3/123 done
Chunk 4/123 done
Chunk 5/123 done
Chunk 6/123 done
Chunk 7/123 done
Chunk 8/123 done
Chunk 9/123 done
Chunk 10/123 done
Chunk 11/123 done
Chunk 12/123 done
Chunk 13/123 done
Chunk 14/123 done
Chunk 15/123 done
Chunk 16/123 done
Chunk 17/123 done
Chunk 18/123 done
Chunk 19/123 done
Chunk 20/123 done
Chunk 21/123 done
Chunk 22/123 done
Chunk 23/123 done
Chunk 24/123 done
Chunk 25/123 done
Chunk 26/123 done
Chunk 27/123 done
Chunk 28/123 done
Chunk 29/123 done
Chunk 30/123 done
Chunk 31/123 done
Chunk 32/123 done
Chunk 33/123 done
Chunk 34/123 done
Chunk 35/123 done
Chunk 36/123 done
Chunk 37/123 done
Chunk 38/123 done
Chunk 39/123 done
Chunk 40/123 done
Chunk 41/123 done
Chunk 42/123 done
Chunk 43/123 done
Chunk 44/123 done
Chunk 45/123 done
Chunk 46/123 done
Chunk 47/123 done
Chunk 48/123 done
Chunk 49/123 done
Chunk 50/123 done
Chunk 51/123 done
Chunk 52/123 done
Chunk 53/123 done
Chunk 54/123 done
Chunk 55/123 done
Chunk 56/123 done
C